In [1]:
import csv
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.gridspec import GridSpec
from collections import defaultdict

In [2]:
STRAINS = ['12B1', '12A1', '12A1-SC1', '12A1-SC2', '12A1-SC3']
#STRAINS = ["12B1", "8020", "8027", "12A1", "12A1-SC1", "12A1-SC2", "12A1-SC3", "13A5","ARC66", "ARC81", "ARC82", "ARC83", "ARC84", "PK2297D4", "PK2298C6", "PK2299B3", "PK2300D3", "UTEX2797", "UTEX2797-SC1", "UTEX2797-SC2", "UTEX2797-SC3", "UTEX2797-SC4", "UTEX2827"]
SCAFFOLD = 'scaf2'
OUTPUT_PATH = f"{SCAFFOLD}_combined_supfig.pdf"

HIGHLIGHT_START = ''
#HIGHLIGHT_START = 3_190_000
#HIGHLIGHT_END   = 3_390_000
HIGHLIGHT_COLOR = "grey"  

In [3]:
def log2_path(strain):
    return (
        "../../../hybrid_diversity/4_read_mapping/3_deduplicated/"
        "2_nuc_and_chloroplast/" + strain + ".log2ratio.tsv"
    )


def allelic_path(strain):
    return (
        "../../../hybrid_diversity/5_snp_analyses/3_aims/3_vcf/"
        "allelic_depth_per_sample/" + strain + ".allelic_depth.tsv"
    )

In [4]:
# ── Plot parameters ────────────────────────────────────────────────────────────
LINEAR_WINDOW  = 10000   # log2 averaging window (bp)
LINEAR_BIN     = 1000    # allelic fraction bin size (bp)
LOG2_CLAMP     = 1.0
DPI            = 200

ROW_HEIGHT     = 1     # inches per strain row (two sub-panels share this)
FIG_WIDTH      = 5      # inches

# ── Colour palettes ───────────────────────────────────
LOG2_CMAP = LinearSegmentedColormap.from_list(
    "log2",
    [(0.000, "#364B9A"),
     (0.1,   "#4A7BB7"),
     (0.3,   "#4A7BB7"),
     (0.500, "#EAECCC"),
     (0.7,   "#DD3D2D"),
     (0.9,   "#DD3D2D"),
     (1.000, "#A50026")],
)

ALLELIC_CMAP = LinearSegmentedColormap.from_list(
    "allelic",
    [(0.00, "#004488"),
     (0.33, "#004488"),
     (0.50, "#DDAA33"),
     (0.67, "#BB5566"),
     (1.00, "#BB5566")],
)

COLOR_BASELINE = "#111111"


def log2_color(val):
    norm = (np.clip(val, -LOG2_CLAMP, LOG2_CLAMP) + LOG2_CLAMP) / (2 * LOG2_CLAMP)
    return LOG2_CMAP(norm)

In [5]:
# ── Data loading ───────────────────────────────────────────────────────────────
def load_log2_linear(path, scaffold, window_size=LINEAR_WINDOW):
    bins = defaultdict(list)
    with open(path) as f:
        reader = csv.DictReader(f, delimiter="\t")
        val_col = [c for c in reader.fieldnames if c not in ("chr", "start", "end")][0]
        for row in reader:
            try:
                if row["chr"] != scaffold:
                    continue
                start = int(row["start"])
                end   = int(row["end"])
                val   = float(row[val_col])
                mid   = (start + end) // 2
                bins[mid // window_size].append((mid, val))
            except (ValueError, KeyError):
                pass
    rows = sorted(
        [(int(np.mean([p for p, v in pts])), np.mean([v for p, v in pts]))
         for pts in bins.values()]
    )
    if not rows:
        return np.array([]), np.array([])
    return (np.array([p for p, v in rows]),
            np.array([v for p, v in rows]))


def load_allelic_linear(path, scaffold, bin_size=LINEAR_BIN):
    bins = defaultdict(list)
    with open(path) as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            try:
                if row["CHROM"] != scaffold:
                    continue
                pos = int(row["POS"])
                val = float(row["A1_frac"])
                bins[pos // bin_size].append((pos, val))
            except (ValueError, KeyError):
                pass
    rows = sorted(
        [(int(np.mean([p for p, v in pts])), np.mean([v for p, v in pts]))
         for pts in bins.values()]
    )
    if not rows:
        return np.array([]), np.array([])
    return (np.array([p for p, v in rows]),
            np.array([v for p, v in rows]))


In [6]:
print(f"Scaffold : {SCAFFOLD}")
print(f"Strains  : {len(STRAINS)}  ({', '.join(STRAINS)})")
print(f"Output   : {OUTPUT_PATH}")

n_strains = len(STRAINS)
fig_height = ROW_HEIGHT * n_strains 

fig = plt.figure(figsize=(FIG_WIDTH, fig_height), facecolor="white")

# Create custom height ratios for tight within-strain and loose between-strain spacing
SPACER       = 0.3    # relative height of between-strain gap rows
INNER_HSPACE = 0.1   # tight space between the two panels within a strain

height_ratios = []
for idx in range(n_strains):
    height_ratios.extend([1, 1])            # log2 and allelic panels
    if idx < n_strains - 1:
        height_ratios.append(SPACER)        # spacer row between strains
height_ratios.append(0.6)                   # colorbar row

n_rows = n_strains * 2 + (n_strains - 1) + 1   # pairs + spacers + colorbar

gs = GridSpec(
    n_rows,
    2,
    figure=fig,
    height_ratios=height_ratios,
    width_ratios=[1, 0.018],
    hspace=INNER_HSPACE,
    wspace=0.03,
)

# Determine shared x-range across all strains
x_max = 0
all_data = {}
missing = []
for strain in STRAINS:
    lp = log2_path(strain)
    ap = allelic_path(strain)
    pl2, vl2 = load_log2_linear(lp, SCAFFOLD)
    pal, val_ = load_allelic_linear(ap, SCAFFOLD)
    all_data[strain] = (pl2, vl2, pal, val_)
    if len(pl2):
        x_max = max(x_max, pl2.max())
    if len(pal):
        x_max = max(x_max, pal.max())

x_max = x_max * 1.02   # small right margin

# Determine global vmax for log2 (used for symmetric y-axis)
all_log2_vals = np.concatenate(
    [v for s, (_, v, _, _) in all_data.items() if len(v)]
)
vmax_log2 = float(np.percentile(np.abs(all_log2_vals), 99)) if len(all_log2_vals) else 1.0
vmax_log2 = max(vmax_log2, 0.3)   # minimum sensible range

# ── Per-strain rows ────────────────────────────────────────────────────────
for i, strain in enumerate(STRAINS):
    pl2, vl2, pal, val_ = all_data[strain]

    row_l2r = i * 3      # each strain occupies 3 rows: log2, allelic, spacer
    row_al  = i * 3 + 1  # (last strain has no spacer but indexing still works)
    
    ax_l2r = fig.add_subplot(gs[row_l2r, 0])
    ax_al  = fig.add_subplot(gs[row_al,  0], sharex=ax_l2r)

    # ── Log2 panel ────────────────────────────────────────────────────────
    if len(pl2):
        colors_l2r = [log2_color(v) for v in vl2]
        ax_l2r.plot(pl2, vl2, color="#cccccc", lw=0.5, zorder=2)
        ax_l2r.scatter(pl2, vl2, c=colors_l2r, s=2,
                       linewidths=0, zorder=3)
    ax_l2r.axhline(0, color=COLOR_BASELINE, lw=0.6, zorder=4)
    ax_l2r.set_yscale("symlog", linthresh=0.5)
    ax_l2r.set_ylim(-vmax_log2 * 1.15, vmax_log2 * 1.15)
    ax_l2r.set_yticks([-1, 0, 1])
    ax_l2r.set_ylabel("LR", fontsize=4, fontfamily="monospace")
    ax_l2r.yaxis.set_major_formatter(matplotlib.ticker.ScalarFormatter())
    ax_l2r.yaxis.set_minor_locator(matplotlib.ticker.NullLocator())
    ax_l2r.spines[["top", "right", "bottom"]].set_visible(False)
    ax_l2r.tick_params(axis="x", labelbottom=False, length=0)
    ax_l2r.tick_params(axis="y", labelsize=5.5)
    ax_l2r.set_xlim(0, x_max)
    ax_l2r.grid(axis="x", which="major", color="lightgrey", lw=0.5, zorder=1)
    ax_l2r.grid(axis="x", which="minor", color="#eeeeee", lw=0.3, zorder=1)
    ax_l2r.xaxis.set_major_locator(matplotlib.ticker.MultipleLocator(1e6))
    ax_l2r.xaxis.set_minor_locator(matplotlib.ticker.MultipleLocator(1e5))

    # ── Allelic panel ─────────────────────────────────────────────────────
    if len(pal):
        colors_al = ALLELIC_CMAP(val_)
        ax_al.scatter(pal, val_, c=colors_al, s=2,
                      linewidths=0, zorder=3)
    ax_al.axhline(0.5, color=COLOR_BASELINE, lw=0.6, zorder=4)
    ax_al.set_ylim(-0.05, 1.05)
    ax_al.set_yticks([0, 0.5, 1])
    ax_al.set_ylabel("AF", fontsize=4, fontfamily="monospace")
    ax_al.yaxis.set_minor_locator(matplotlib.ticker.NullLocator())
    ax_al.spines[["top", "right"]].set_visible(False)
    ax_al.tick_params(axis="y", labelsize=5.5)
    ax_al.set_xlim(0, x_max)
    ax_al.grid(axis="x", which="major", color="lightgrey", lw=0.5, zorder=1)
    ax_al.grid(axis="x", which="minor", color="#eeeeee", lw=0.3, zorder=1)
    ax_al.xaxis.set_major_locator(matplotlib.ticker.MultipleLocator(1e6))
    ax_al.xaxis.set_minor_locator(matplotlib.ticker.MultipleLocator(1e5))

    # Only show x tick labels on the last strain's allelic panel
    is_last = (i == n_strains - 1)
    if is_last:
        ax_al.xaxis.set_major_formatter(
            matplotlib.ticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f} Mb")
        )
        ax_al.tick_params(axis="x", labelsize=4)
    else:
        ax_al.tick_params(axis="x", labelbottom=False, length=0)
        ax_al.spines["bottom"].set_visible(False)

    if HIGHLIGHT_START != '':
        ax_l2r.axvspan(HIGHLIGHT_START, HIGHLIGHT_END, color=HIGHLIGHT_COLOR, alpha=0.2, zorder=1)
        ax_al.axvspan(HIGHLIGHT_START, HIGHLIGHT_END, color=HIGHLIGHT_COLOR, alpha=0.2, zorder=1)

# Add strain labels vertically on the left, centered across both panels
for i, strain in enumerate(STRAINS):
    ax_l2r = fig.axes[i * 2]      # Log2 subplot for this strain
    ax_al = fig.axes[i * 2 + 1]   # Allelic subplot for this strain
    
    # Get the y-position at the center of both panels combined
    y_center = (ax_l2r.get_position().y0 + ax_al.get_position().y1) / 2
    
    fig.text(0, y_center, strain, 
             fontsize=6, fontfamily="monospace",
             rotation=90, ha="right", va="center",
             transform=fig.transFigure)

for ax in fig.axes:
    ax.tick_params(labelsize=4)
    
fig.savefig(OUTPUT_PATH, dpi=DPI, bbox_inches="tight", facecolor="white")
print(f"Saved → {OUTPUT_PATH}")
plt.close(fig)




Scaffold : scaf2
Strains  : 5  (12B1, 12A1, 12A1-SC1, 12A1-SC2, 12A1-SC3)
Output   : scaf2_combined_supfig.pdf
Saved → scaf2_combined_supfig.pdf
